In [1]:
!pip install scipy numpy --quiet

In [ ]:
import math

def sample_size_two_proportion(p1, delta, alpha=0.05, power=0.80):
    z_alpha = 1.959964  # two-sided
    z_beta = 0.841621
    p2 = p1 + delta
    p_bar = (p1 + p2) / 2
    numerator = (
        z_alpha * math.sqrt(2 * p_bar * (1 - p_bar)) +
        z_beta * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))
    ) ** 2
    n = numerator / (delta ** 2)
    return math.ceil(n)

p1 = 0.4124  # real baseline from Phase 2
daily_checkout_starts = 2260 / 92  # actual observed rate

print(f"{'MDE':>6} | {'p2':>8} | {'n/arm':>8} | {'total':>8} | {'est. days':>10}")
for delta in [0.03, 0.05, 0.07, 0.10]:
    n = sample_size_two_proportion(p1, delta)
    total = n * 2
    days = total / daily_checkout_starts
    print(f"{delta:>6.2f} | {p1+delta:>8.4f} | {n:>8} | {total:>8} | {days:>10.1f}")

   MDE |       p2 |    n/arm |    total |  est. days
  0.03 |   0.4424 |     4268 |     8536 |      347.5
  0.05 |   0.4624 |     1544 |     3088 |      125.7
  0.07 |   0.4824 |      791 |     1582 |       64.4
  0.10 |   0.5124 |      390 |      780 |       31.8


In [3]:
import numpy as np
from scipy.stats import norm

np.random.seed(42)

# SIMULATED DATA — assumed effect size, not observed from BigQuery
n_per_arm = 791
control_rate = 0.4124   # real baseline
treatment_rate = 0.4824

control_conversions = np.random.binomial(1, control_rate, n_per_arm)
treatment_conversions = np.random.binomial(1, treatment_rate, n_per_arm)

p1_obs = control_conversions.mean()
p2_obs = treatment_conversions.mean()
p_pool = (control_conversions.sum() + treatment_conversions.sum()) / (2 * n_per_arm)
se = np.sqrt(p_pool * (1 - p_pool) * (2 / n_per_arm))
z_stat = (p2_obs - p1_obs) / se
p_value = 2 * (1 - norm.cdf(abs(z_stat)))

print("SIMULATED RESULT — assumed effect size, not real experimental data")
print(f"Simulated control rate: {p1_obs:.4f}")
print(f"Simulated treatment rate: {p2_obs:.4f}")
print(f"Z-statistic: {z_stat:.3f}")
print(f"P-value: {p_value:.4f}")

SIMULATED RESULT — assumed effect size, not real experimental data
Simulated control rate: 0.4109
Simulated treatment rate: 0.4968
Z-statistic: 3.434
P-value: 0.0006
